# Indian Startup Legal Assistant: Data Preparation and Indexing Pipeline

This notebook implements the complete data ingestion, text preprocessing, chunking, and dual indexing (Dense FAISS + Lexical BM25) pipeline for Indian statutory legal documents.

### Pipeline Overview
1. **Environment & Setup**: Dependencies, Google Drive mounting, and imports.
2. **Configuration & Directory Setup**: Paths and hyperparameter definitions.
3. **Raw Legal Data Ingestion**: Loading and parsing structured legal JSON files.
4. **Document Extraction & Metadata Normalization**: Converting raw sections into standard retrieval documents.
5. **Data Quality Checks**: Validation for empty records and length distribution.
6. **Base Document Artifacts Persistence**: Baseline document pickle, model caching, FAISS index, and BM25 index.
7. **Document Length Analysis**: Empirical statistical analysis of section lengths motivating chunking.
8. **Legal Text Chunking**: Boundary-aware sliding window segmentation with sentence and clause preservation.
9. **Dense Vector Indexing (FAISS)**: Generating normalized dense embeddings using `all-MiniLM-L6-v2` and indexing with FAISS `IndexFlatIP`.
10. **Lexical Indexing (BM25)**: Tokenization and building the BM25 Okapi index over chunked documents.
11. **Artifact Verification & Integrity Checks**: Ensuring alignment between indices, corpus entries, and persistent storage.

## 1. Environment & Setup

Install required dependencies and configure the execution environment.

In [ ]:
!pip -q install sentence-transformers faiss-cpu rank-bm25 langgraph langchain-core google-genai

In [ ]:
from google.colab import drive

# Mount Google Drive to persist generated artifacts
drive.mount('/content/drive')

In [ ]:
import os
import json
import re
import pickle
from pathlib import Path
from typing import Dict, List, Any, Optional

import numpy as np
import torch
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

# Check hardware acceleration
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device   :", torch.cuda.get_device_name(0))

## 2. Configuration & Paths

Define source data directories, output destination paths, embedding model names, and chunking hyperparameters.

In [ ]:
# Storage Paths
DATA_FOLDER = Path("/content/drive/MyDrive/Gemma DTU Hackathon/JSON files")
SAVE_DIR = Path("/content/drive/MyDrive/Gemma DTU Outputs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Model & Chunking Parameters
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
CHUNK_SIZE = 1100
OVERLAP = 160
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Data Folder   : {DATA_FOLDER}")
print(f"Save Directory: {SAVE_DIR}")
print(f"Compute Device: {DEVICE}")

## 3. Raw Legal Data Ingestion

Load raw legal JSON files containing acts, chapters, and statutory sections.

In [ ]:
json_files = sorted(DATA_FOLDER.glob("*.json"))

documents = {}
for file in json_files:
    with open(file, "r", encoding="utf-8") as f:
        documents[file.stem] = json.load(f)

print(f"Loaded {len(documents)} JSON legal document collections.")

## 4. Document Extraction & Metadata Normalization

Extract statutory sections from raw JSON structures and format them into normalized retrieval documents with structured metadata and formatted embedding text.

In [ ]:
retrieval_documents = []

for records in documents.values():
    for record in records:
        act_name = record.get("act_name", "")
        chapter = record.get("chapter", "")
        chapter_title = record.get("chapter_title", "")
        section_number = record.get("section_number", "")
        section_title = record.get("section_title", "")
        section_text = record.get("section_text", "").strip()

        embedding_text = f"""
Act: {act_name}

Chapter: {chapter}
Chapter Title: {chapter_title}

Section: {section_number}
Section Title: {section_title}

Content:
{section_text}
""".strip()

        chunk_id = (
            re.sub(r"[^A-Za-z0-9_]", "", act_name.replace(" ", "_"))
            + "_"
            + str(section_number)
        )

        retrieval_documents.append({
            "chunk_id": chunk_id,
            "text": section_text,
            "embedding_text": embedding_text,
            "metadata": {
                "act_name": act_name,
                "source_file": record.get("source_file"),
                "chapter": chapter,
                "chapter_title": chapter_title,
                "section_number": section_number,
                "section_title": section_title
            }
        })

print(f"Created {len(retrieval_documents)} retrieval documents.")

## 5. Data Quality Checks & Base Artifact Persistence

Validate document contents, verify no empty sections exist, and persist base documents and embedding model locally.

In [ ]:
empty = [doc for doc in retrieval_documents if len(doc["text"]) == 0]
print(f"Empty chunks: {len(empty)}")

lengths = [len(doc["text"]) for doc in retrieval_documents]
print(f"Total chunks        : {len(lengths)}")
print(f"Average characters  : {sum(lengths)/len(lengths):.1f}")
print(f"Shortest chunk      : {min(lengths)}")
print(f"Longest chunk       : {max(lengths)}")

# Persist base retrieval documents
with open(SAVE_DIR / "retrieval_documents.pkl", "wb") as f:
    pickle.dump(retrieval_documents, f)

print(f"Retrieval documents saved to: {SAVE_DIR / 'retrieval_documents.pkl'}")

In [ ]:
# Download and cache SentenceTransformer model locally
embedding_model = SentenceTransformer(
    f"sentence-transformers/{EMBEDDING_MODEL_NAME}",
    device=DEVICE
)

embedding_model.save(
    str(SAVE_DIR / EMBEDDING_MODEL_NAME)
)

print(f"Model saved successfully to {SAVE_DIR / EMBEDDING_MODEL_NAME}")

## 6. Base (Unchunked) FAISS & BM25 Indexing Pipeline

Build and save baseline unchunked vector and keyword indices for complete artifact compatibility.

In [ ]:
# Generate base embeddings
corpus = [
    doc["embedding_text"]
    for doc in retrieval_documents
]

print(f"Corpus size: {len(corpus)}")

embeddings = embedding_model.encode(
    corpus,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Base embeddings shape:", embeddings.shape)

# Build and persist base FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(f"Indexed {index.ntotal} base chunks into FAISS.")

np.save(SAVE_DIR / "embeddings.npy", embeddings)
faiss.write_index(index, str(SAVE_DIR / "faiss.index"))

# Save base configuration
config = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "embedding_dimension": int(dimension),
    "normalize_embeddings": True,
    "faiss_index": "IndexFlatIP"
}

with open(SAVE_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=4)

print("Base embeddings, FAISS index, and config saved.")

In [ ]:
# Build and persist base BM25 index
bm25_corpus = []

for doc in retrieval_documents:
    searchable_text = f"""
    {doc["metadata"]["act_name"]}
    {doc["metadata"]["section_title"]}
    {doc["text"]}
    """
    tokens = re.findall(r"\w+", searchable_text.lower())
    bm25_corpus.append(tokens)

print(f"Prepared {len(bm25_corpus)} BM25 documents.")

bm25 = BM25Okapi(bm25_corpus)

with open(SAVE_DIR / "bm25.pkl", "wb") as f:
    pickle.dump(bm25, f)

with open(SAVE_DIR / "bm25_corpus.pkl", "wb") as f:
    pickle.dump(bm25_corpus, f)

print("Base BM25 index and corpus saved.")

## 7. Document Length Analysis & Chunking Rationale

Analyze the length distribution of statutory sections to identify long sections that exceed embedding token limits, establishing the empirical need for boundary-aware chunking.

In [ ]:
lengths_arr = np.array([
    len(doc["text"])
    for doc in retrieval_documents
])

print("TOTAL DOCUMENTS:", len(lengths_arr))
print("Embedding model max sequence length:", embedding_model.max_seq_length)

print("\nDOCUMENT LENGTH STATISTICS")
print("Minimum        :", np.min(lengths_arr))
print("Median         :", np.median(lengths_arr))
print("90th percentile:", np.percentile(lengths_arr, 90))
print("95th percentile:", np.percentile(lengths_arr, 95))
print("99th percentile:", np.percentile(lengths_arr, 99))
print("Maximum        :", np.max(lengths_arr))

print("\nLONGEST DOCUMENTS:")
longest = sorted(
    retrieval_documents,
    key=lambda x: len(x["text"]),
    reverse=True
)[:10]

for doc in longest:
    print(
        f"{len(doc['text']):5d} chars | "
        f"{doc['metadata']['act_name']} | "
        f"Section {doc['metadata']['section_number']} | "
        f"{doc['metadata']['section_title']}"
    )

## 8. Legal Text Chunking Implementation

Implement `split_legal_text` using a boundary-aware sliding window that prioritizes sentence (`. `), semicolon (`; `), and word boundaries, followed by constructing `chunked_documents` with parent section traceability.

In [ ]:
def split_legal_text(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):

    text = re.sub(r"\s+", " ", str(text)).strip()

    if not text:
        return []

    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0

    while start < len(text):

        end = min(start + chunk_size, len(text))

        if end < len(text):

            min_boundary = start + int(chunk_size * 0.65)

            candidates = [
                text.rfind(". ", min_boundary, end),
                text.rfind("; ", min_boundary, end),
                text.rfind(" ", min_boundary, end)
            ]

            boundary = max(candidates)

            if boundary > start:
                end = boundary + 1

        piece = text[start:end].strip()

        if piece:
            chunks.append(piece)

        if end >= len(text):
            break

        start = max(start + 1, end - overlap)

    return chunks


chunked_documents = []

for doc in retrieval_documents:

    metadata = doc["metadata"]

    text_chunks = split_legal_text(doc["text"])

    for part_number, text_chunk in enumerate(text_chunks, 1):

        chunk_id = (
            f"{doc['chunk_id']}_part_{part_number:03d}"
        )

        embedding_text = f"""
Act: {metadata["act_name"]}
Chapter: {metadata.get("chapter_title", "")}
Section: {metadata["section_number"]}
Section Title: {metadata["section_title"]}

Legal Content:
{text_chunk}
"""

        chunked_documents.append({
            "chunk_id": chunk_id,
            "parent_chunk_id": doc["chunk_id"],
            "text": text_chunk,
            "embedding_text": embedding_text,
            "metadata": {
                **metadata,
                "parent_chunk_id": doc["chunk_id"],
                "part_number": part_number
            }
        })


print("Original documents :", len(retrieval_documents))
print("New retrieval chunks:", len(chunked_documents))

print(
    "Average chunk length:",
    round(
        np.mean([len(doc["text"]) for doc in chunked_documents]),
        2
    )
)

# Persist chunked retrieval documents
with open(SAVE_DIR / "chunked_retrieval_documents.pkl", "wb") as f:
    pickle.dump(chunked_documents, f)

print(f"Chunked retrieval documents saved to {SAVE_DIR / 'chunked_retrieval_documents.pkl'}")

## 9. Dense Vector Indexing on Chunks (FAISS)

Generate dense normalized vector embeddings for chunked documents using `SentenceTransformer` and construct a FAISS inner-product (`IndexFlatIP`) index.

In [ ]:
chunked_corpus = [
    doc["embedding_text"]
    for doc in chunked_documents
]

chunked_embeddings = embedding_model.encode(
    chunked_corpus,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

chunked_embeddings = np.asarray(
    chunked_embeddings,
    dtype=np.float32
)

dimension = chunked_embeddings.shape[1]

chunked_index = faiss.IndexFlatIP(dimension)
chunked_index.add(chunked_embeddings)

print("Embeddings shape:", chunked_embeddings.shape)
print("FAISS vectors   :", chunked_index.ntotal)

# Persist chunked dense artifacts
np.save(
    SAVE_DIR / "chunked_embeddings.npy",
    chunked_embeddings
)

faiss.write_index(
    chunked_index,
    str(SAVE_DIR / "chunked_faiss.index")
)

print("Chunked embeddings and FAISS index saved.")

## 10. Lexical Indexing on Chunks (BM25)

Prepare a tokenized BM25 corpus from chunked document text and construct the BM25 Okapi index for hybrid keyword retrieval.

In [ ]:
# Prepare tokenised corpus for BM25
chunked_bm25_corpus = []

for doc in chunked_documents:

    searchable_text = f"""
    {doc["metadata"]["act_name"]}
    {doc["metadata"]["section_title"]}
    {doc["text"]}
    """

    tokens = re.findall(r"\w+", searchable_text.lower())
    chunked_bm25_corpus.append(tokens)


# Build BM25 index
chunked_bm25 = BM25Okapi(chunked_bm25_corpus)

# Save BM25 artifacts
with open(SAVE_DIR / "chunked_bm25.pkl", "wb") as f:
    pickle.dump(chunked_bm25, f)

with open(SAVE_DIR / "chunked_bm25_corpus.pkl", "wb") as f:
    pickle.dump(chunked_bm25_corpus, f)

print("BM25 documents:", len(chunked_bm25_corpus))
print("BM25 index built and saved.")

## 11. Artifact Verification & Integrity Checks

Verify that all generated vector and keyword indexing artifacts exist on disk and possess matching dimensionalities and document counts.

In [ ]:
print("=" * 60)
print("ARTIFACT INTEGRITY VALIDATION")
print("=" * 60)
print("FAISS total vectors      :", chunked_index.ntotal)
print("Chunked documents count  :", len(chunked_documents))
print("BM25 corpus entries count:", len(chunked_bm25_corpus))
print("Embedding dimension      :", dimension)

assert chunked_index.ntotal == len(chunked_documents), "Mismatch between FAISS index and documents count!"
assert len(chunked_bm25_corpus) == len(chunked_documents), "Mismatch between BM25 corpus and documents count!"

expected_files = [
    "all-MiniLM-L6-v2",
    "retrieval_documents.pkl",
    "embeddings.npy",
    "faiss.index",
    "config.json",
    "bm25.pkl",
    "bm25_corpus.pkl",
    "chunked_retrieval_documents.pkl",
    "chunked_embeddings.npy",
    "chunked_faiss.index",
    "chunked_bm25.pkl",
    "chunked_bm25_corpus.pkl"
]

print("\nVerifying saved files on disk:")
for fname in expected_files:
    fpath = SAVE_DIR / fname
    status = "EXISTS" if fpath.exists() else "MISSING"
    print(f"[{status:13s}] {fname}")

print("\nIndexing and artifact generation completed successfully.")